# MAP Groundwater Reconstruction

This notebook rebuilds the main 2011-2023 reconstruction, applies the regional storage correction, and propagates PI75 model uncertainty.

In [ ]:
from pathlib import Path
import subprocess
import sys
import time

import pandas as pd

def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    candidates = [start, *start.parents]
    for cand in candidates:
        if (cand / "outputs").exists() and (cand / "notebooks").exists():
            return cand
    raise FileNotFoundError("Could not locate repo root containing 'outputs' and 'notebooks'.")

def display_path(path: Path | str) -> str:
    p = Path(path)
    try:
        return str(p.resolve().relative_to(REPO_ROOT))
    except ValueError:
        return str(p)


def scrub_project_paths(text: str) -> str:
    root = str(REPO_ROOT)
    home = str(Path.home())
    cleaned = text
    for prefix in {root, root.replace('\\', '/')}:
        cleaned = cleaned.replace(prefix + '\\', '')
        cleaned = cleaned.replace(prefix + '/', '')
        cleaned = cleaned.replace(prefix, '.')
    for prefix in {home, home.replace('\\', '/')}:
        cleaned = cleaned.replace(prefix, '<HOME>')
    return cleaned


def run_project_step(args: list[str], label: str) -> None:
    print(f"Running {label}...")
    t0 = time.time()
    result = subprocess.run(args, cwd=REPO_ROOT, text=True, capture_output=True)
    if result.stdout:
        print(scrub_project_paths(result.stdout), end="")
    if result.stderr:
        print(scrub_project_paths(result.stderr), end="")
    if result.returncode != 0:
        raise RuntimeError(f"{label} failed with exit code {result.returncode}. See tool output above.")
    print(f"Finished {label} in {(time.time() - t0) / 60:.1f} min")

REPO_ROOT = find_repo_root()
RECON_ROOT = REPO_ROOT / "outputs" / "RECON_MAIN_2011_2023"
RECON_MATRIX_PATH = RECON_ROOT / "reconstruction" / "wtd_reconstructed_matrix.npy"
MONTH_INDEX_PATH = RECON_ROOT / "metadata" / "month_index.csv"
UNC_MATRIX_PATH = RECON_ROOT / "model_uncertainty" / "monthly_model_uncertainty_radius_matrix.npy"
UNC_METADATA_PATH = RECON_ROOT / "model_uncertainty" / "model_uncertainty_metadata.json"

run_project_step(
    [
        sys.executable,
        str(REPO_ROOT / "tools" / "run_recon.py"),
        "--out-dir",
        str(RECON_ROOT),
    ],
    "WTD reconstruction",
)
run_project_step(
    [
        sys.executable,
        str(REPO_ROOT / "tools" / "recon_mass_correction.py"),
        str(RECON_ROOT),
        str(RECON_ROOT),
    ],
    "regional storage-mass correction",
)

month_index = pd.read_csv(MONTH_INDEX_PATH)
MONTH_COL = "month" if "month" in month_index.columns else "month_label"

print(f"Repo root: {display_path(REPO_ROOT)}")
print(f"Reconstruction root: {display_path(RECON_ROOT)}")
print(f"Reconstruction matrix: {display_path(RECON_MATRIX_PATH)}")
print(f"Month column used: {MONTH_COL}")
run_project_step(
    [
        sys.executable,
        str(REPO_ROOT / "tools" / "propagate_uncertainty.py"),
        str(RECON_ROOT),
    ],
    "model uncertainty propagation",
)

if not UNC_MATRIX_PATH.exists():
    raise FileNotFoundError(f"Uncertainty matrix not found: {display_path(UNC_MATRIX_PATH)}")
if not UNC_METADATA_PATH.exists():
    raise FileNotFoundError(f"Uncertainty metadata not found: {display_path(UNC_METADATA_PATH)}")
print(f"Uncertainty matrix: {display_path(UNC_MATRIX_PATH)}")
print(f"Uncertainty metadata: {display_path(UNC_METADATA_PATH)}")
